In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog, scrolledtext
import pandas as pd
import numpy as np
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from ngboost import NGBRegressor
from ngboost.distns import Normal
from sklearn.model_selection import train_test_split, KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats
import threading
from datetime import datetime
import json
import os
import matplotlib.pyplot as plt
from matplotlib import cm
import warnings
import re

warnings.filterwarnings('ignore')

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Install with: pip install shap")

# ============================================================
# [FIX 1] TERMINOLOGY CHANGE
# ============================================================
# Previous (NOT defensible): "Scientifically Justified Risk Thresholds"
# New: "Quartile-Based Categorization"
#
# These quartile cut points provide a RELATIVE categorization
# within the dataset. They are NOT externally validated thresholds.
# This distinction must be preserved in the paper.
# ============================================================


class TargetPredictor:
    def __init__(self):
        self.setup_style()
        self.setup_paths()
        self.load_data()
        self.train_test_split()
        self.train_model()
        self.run_cross_validation()
        self.compute_prediction_uncertainty()   # [FIX 2] renamed
        self.compute_shap_values()
        self.compute_correlations()
        self.create_gui()

    # ------------------------------------------------------------------
    # Style / paths
    # ------------------------------------------------------------------
    def setup_style(self):
        self.colors = {
            'primary': '#2E4053',
            'secondary': '#6A1E55',
            'accent': '#D4A373',
            'background': '#F8F9FA',
            'card_bg': '#FFFFFF',
            'success': '#2D6A4F',
            'warning': '#B9770E',
            'danger': '#B03A2E',
            'sandy': '#D4A373',
            'soil_brown': '#8B5A2B',
            'rf_green': '#2E8B57',
            'pso_purple': '#800080',
            'strength_blue': '#2D6A4F',
            'ngboost_orange': '#FF6B35',
        }

    def setup_paths(self):
        self.data_path = r"D:\2026 Work\My Papers\Gulzar\Data\Data.csv"
        self.save_dir = r"D:\2026 Work\My Papers\Gulzar\GUI"
        os.makedirs(self.save_dir, exist_ok=True)
        self.history_file = os.path.join(self.save_dir, "prediction_history.json")

    def clean_column_names(self, columns):
        cleaned = []
        for col in columns:
            c = re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col))
            c = re.sub(r'[ _]+', '_', c).strip('_')
            cleaned.append(c if c else f'feature_{len(cleaned)}')
        return cleaned

    # ------------------------------------------------------------------
    # Data
    # ------------------------------------------------------------------
    def load_data(self):
        try:
            print(f"Loading data from: {self.data_path}")
            self.df = pd.read_csv(self.data_path, encoding='ISO-8859-1')
            print(f"Dataset shape: {self.df.shape}")
            self.df.columns = self.clean_column_names(self.df.columns)

            # Target variable detection - specifically for "CS (MPa)"
            target_col = None
            target_candidates = ['CS (MPa)', 'CS_MPa', 'CS_MPa_', 'CS', 'cs', 'CS(MPa)']
            
            for t in target_candidates:
                if t in self.df.columns:
                    target_col = t
                    break
                # Try normalized matching
                t_norm = t.lower().replace('_', '').replace(' ', '').replace('(', '').replace(')', '')
                for col in self.df.columns:
                    col_norm = col.lower().replace('_', '').replace(' ', '').replace('(', '').replace(')', '')
                    if t_norm == col_norm:
                        target_col = col
                        break
                if target_col:
                    break

            # Fallback: look for columns containing 'cs' or 'mpa'
            if target_col is None:
                for col in self.df.columns:
                    col_lower = col.lower()
                    if 'cs' in col_lower and 'mpa' in col_lower:
                        target_col = col
                        break
                    elif 'cs' in col_lower:
                        target_col = col
                        break

            if target_col is None:
                # Last resort: use last column
                target_col = self.df.columns[-1]
                print(f"Warning: Target column not found by name, using last column: '{target_col}'")

            self.target_name = target_col
            print(f"Target variable: '{self.target_name}'")

            self.X = self.df.drop(columns=[target_col])
            self.y = self.df[target_col]
            self.feature_names = self.X.columns.tolist()

            # Remove any non-numeric columns from features
            numeric_cols = []
            for f in self.feature_names:
                if pd.api.types.is_numeric_dtype(self.X[f]):
                    numeric_cols.append(f)
                else:
                    print(f"Dropping non-numeric feature: {f}")
            self.X = self.X[numeric_cols]
            self.feature_names = numeric_cols

            self.feature_stats = {}
            for f in self.feature_names:
                self.feature_stats[f] = {
                    'min': float(self.X[f].min()), 
                    'max': float(self.X[f].max()),
                    'mean': float(self.X[f].mean()), 
                    'std': float(self.X[f].std()),
                    'median': float(self.X[f].median()),
                    'q1': float(self.X[f].quantile(0.25)), 
                    'q3': float(self.X[f].quantile(0.75))
                }

            print(f"Features: {len(self.feature_names)}")
            print(f"Target stats: mean={self.y.mean():.4f}, std={self.y.std():.4f}, "
                  f"min={self.y.min():.4f}, max={self.y.max():.4f}")

        except Exception as e:
            messagebox.showerror("Data Loading Error", str(e))
            raise

    def train_test_split(self):
        print("\n" + "="*60)
        print("TRAIN-TEST SPLIT (80-20)")
        print("="*60)
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.20, random_state=42
        )
        print(f"Training: {len(self.X_train)} samples (80%)")
        print(f"Test:     {len(self.X_test)} samples (20%, independent)")

        # [FIX 1] Quartile-based categorization (relative, not absolute)
        # ------------------------------------------------------------------
        # These quartiles are computed on the TRAINING data only to avoid
        # any leakage from the test set. They provide a RELATIVE ranking
        # within the dataset. They are not externally validated thresholds.
        # ------------------------------------------------------------------
        self.quartile_thresholds = {
            'q25': float(self.y_train.quantile(0.25)),
            'q50': float(self.y_train.quantile(0.50)),
            'q75': float(self.y_train.quantile(0.75)),
        }
        print(f"\nQuartile-based Categorization (training data):")
        print(f"  Q1 (25th percentile): {self.quartile_thresholds['q25']:.4f}")
        print(f"  Q2 (median, 50th):    {self.quartile_thresholds['q50']:.4f}")
        print(f"  Q3 (75th percentile): {self.quartile_thresholds['q75']:.4f}")
        print("  NOTE: These are RELATIVE categories within the dataset,")
        print("        not externally validated thresholds.")

    # ------------------------------------------------------------------
    # Model
    # ------------------------------------------------------------------
    def train_model(self):
        print("\n" + "="*60)
        print("TRAINING NGBOOST MODEL")
        print("="*60)

        # NGBoost hyperparameters (from user specification)
        self.hyperparams = {
            'verbose_eval': False,
            'verbose': False,
            'tol': 1.0e-05,
            'natural_gradient': True,
            'n_estimators': 700,
            'minibatch_frac': 0.7,
            'learning_rate': 0.07,
            'col_sample': 1.0,
            'random_state': 42,
        }
        for k, v in self.hyperparams.items():
            print(f"  {k:25}: {v}")

        # NGBoost with Normal distribution for regression
        self.model = NGBRegressor(
            Dist=Normal,
            **self.hyperparams
        )
        self.model.fit(self.X_train.values, self.y_train.values)
        print("Model trained.")

        self.y_train_pred = self.model.predict(self.X_train.values)
        self.y_test_pred = self.model.predict(self.X_test.values)

        # [FIX 3] Permutation importance evaluated on TEST set (out-of-sample)
        # ------------------------------------------------------------------
        # Rationale: importance computed on training data can overstate
        # out-of-sample relevance. Using the test set provides a more
        # honest out-of-sample estimate. The test set is used ONLY for
        # post-hoc interpretation, never for model selection.
        # ------------------------------------------------------------------
        print("\nComputing permutation feature importance (out-of-sample, test set)...")
        perm = permutation_importance(
            self.model, self.X_test.values, self.y_test.values,
            n_repeats=30, random_state=42, n_jobs=-1
        )
        self.feature_importance = pd.DataFrame({
            'feature': self.feature_names,
            'importance': perm.importances_mean,
            'std': perm.importances_std
        }).sort_values('importance', ascending=False)

        print("\nTop 5 features (permutation, out-of-sample):")
        for i, (_, r) in enumerate(self.feature_importance.head(5).iterrows(), 1):
            print(f"  {i}. {r['feature'][:25]:25}: {r['importance']:.4f} ± {r['std']:.4f}")

        self._calculate_metrics()

    def _calculate_metrics(self):
        self.r2_train = r2_score(self.y_train, self.y_train_pred)
        self.rmse_train = np.sqrt(mean_squared_error(self.y_train, self.y_train_pred))
        self.mae_train = mean_absolute_error(self.y_train, self.y_train_pred)

        self.r2_test = r2_score(self.y_test, self.y_test_pred)
        self.rmse_test = np.sqrt(mean_squared_error(self.y_test, self.y_test_pred))
        self.mae_test = mean_absolute_error(self.y_test, self.y_test_pred)

        # [FIX 7] MAPE is supplementary only
        mask = np.abs(self.y_test) > 1e-6
        if mask.any():
            self.mape_test = float(np.mean(
                np.abs((self.y_test[mask] - self.y_test_pred[mask]) / self.y_test[mask])
            ) * 100)
        else:
            self.mape_test = np.nan

        print(f"\nPerformance:")
        print(f"  Train  R²={self.r2_train:.4f}  RMSE={self.rmse_train:.4f}  MAE={self.mae_train:.4f}")
        print(f"  Test   R²={self.r2_test:.4f}  RMSE={self.rmse_test:.4f}  MAE={self.mae_test:.4f}")
        print(f"  Test   MAPE={self.mape_test:.2f}% (supplementary metric)")

    # ------------------------------------------------------------------
    # Cross-validation
    # ------------------------------------------------------------------
    def run_cross_validation(self):
        # [FIX 4] Manual CV loop (NGBoost doesn't work with sklearn's cross_validate)
        print("\n" + "="*60)
        print("5-FOLD CROSS-VALIDATION (on training data only)")
        print("="*60)
        try:
            kf = KFold(n_splits=5, shuffle=True, random_state=42)

            r2_scores, rmse_scores, mae_scores = [], [], []
            for train_idx, val_idx in kf.split(self.X_train):
                X_tr = self.X_train.iloc[train_idx].values
                y_tr = self.y_train.iloc[train_idx].values
                X_val = self.X_train.iloc[val_idx].values
                y_val = self.y_train.iloc[val_idx].values

                fold_model = NGBRegressor(Dist=Normal, **self.hyperparams)
                fold_model.fit(X_tr, y_tr)
                y_val_pred = fold_model.predict(X_val)

                r2_scores.append(r2_score(y_val, y_val_pred))
                rmse_scores.append(np.sqrt(mean_squared_error(y_val, y_val_pred)))
                mae_scores.append(mean_absolute_error(y_val, y_val_pred))

            self.cv_results = {
                'r2_scores': r2_scores,
                'r2_mean': float(np.mean(r2_scores)),
                'r2_std': float(np.std(r2_scores)),
                'rmse_scores': rmse_scores,
                'rmse_mean': float(np.mean(rmse_scores)),
                'rmse_std': float(np.std(rmse_scores)),
                'mae_scores': mae_scores,
                'mae_mean': float(np.mean(mae_scores)),
                'mae_std': float(np.std(mae_scores)),
            }

            print(f"  R²   : {self.cv_results['r2_mean']:.4f} ± {self.cv_results['r2_std']:.4f}")
            print(f"  RMSE : {self.cv_results['rmse_mean']:.4f} ± {self.cv_results['rmse_std']:.4f}")
            print(f"  MAE  : {self.cv_results['mae_mean']:.4f} ± {self.cv_results['mae_std']:.4f}")

        except Exception as e:
            print(f"CV failed: {e}")
            import traceback
            traceback.print_exc()
            self.cv_results = None

    # ------------------------------------------------------------------
    # [FIX 2] Approximate 95% PREDICTION INTERVAL (not "confidence interval")
    # ------------------------------------------------------------------
    def compute_prediction_uncertainty(self):
        """
        Approximate 95% prediction interval based on training-set residual
        variability.

        IMPORTANT: This is NOT a rigorous model-specific prediction interval.
        It uses a homoscedastic assumption (constant interval width across
        the prediction range). For a nonlinear model such as NGBoost this
        is a simplification. The interval should be reported as:

            'Approximate 95% prediction interval based on training-set
             residual variability'

        The empirical coverage on the test set is reported for transparency
        but does NOT guarantee 95% coverage.
        """
        print("\n" + "="*60)
        print("APPROXIMATE 95% PREDICTION INTERVAL")
        print("(based on training residual variability)")
        print("="*60)

        train_residuals = self.y_train.values - self.y_train_pred
        self.residual_std = float(np.std(train_residuals, ddof=1))
        n_train = len(self.y_train)

        # Standard error of an individual prediction (approx.)
        self.se_pred = self.residual_std * np.sqrt(1.0 + 1.0/n_train)

        # t-based half width at 95%
        self.t_crit = float(stats.t.ppf(0.975, df=n_train - 1))
        self.pi_half_width = self.t_crit * self.se_pred   # [FIX 2] renamed

        self.test_pi_lower = self.y_test_pred - self.pi_half_width
        self.test_pi_upper = self.y_test_pred + self.pi_half_width

        inside = ((self.y_test.values >= self.test_pi_lower) &
                  (self.y_test.values <= self.test_pi_upper)).sum()
        self.pi_empirical_coverage = inside / len(self.y_test) * 100

        print(f"  Residual std (train):    {self.residual_std:.4f}")
        print(f"  t critical (95%):        {self.t_crit:.4f}")
        print(f"  PI half width:           ±{self.pi_half_width:.4f}")
        print(f"  Empirical test coverage: {self.pi_empirical_coverage:.1f}%")
        print("  NOTE: Interval width is constant (homoscedastic approximation).")
        print("        Reported as 'approximate 95% prediction interval'.")

        # NGBoost native uncertainty (predictive distribution)
        try:
            self.y_test_dist = self.model.pred_dist(self.X_test.values)
            self.y_test_std_native = self.y_test_dist.params['scale']
            print(f"\n  NGBoost native predictive std (test): mean={np.mean(self.y_test_std_native):.4f}")
        except Exception as e:
            print(f"  NGBoost native uncertainty not available: {e}")
            self.y_test_dist = None
            self.y_test_std_native = None

    # ------------------------------------------------------------------
    # SHAP
    # ------------------------------------------------------------------
    def compute_shap_values(self):
        print("\n" + "="*60)
        print("SHAP ANALYSIS (post-hoc interpretation on test set)")
        print("="*60)
        if not SHAP_AVAILABLE:
            print("SHAP not available.")
            self.shap_explainer = None
            self.shap_values = None
            self.shap_importance = None
            return
        try:
            # NGBoost doesn't have a direct TreeExplainer; use KernelExplainer
            # Sample background data for efficiency
            background = shap.sample(self.X_train, min(100, len(self.X_train)))
            
            # Use a wrapper for prediction
            def model_predict(X):
                return self.model.predict(X)
            
            self.shap_explainer = shap.KernelExplainer(
                model_predict, background.values
            )
            
            # Compute SHAP values on a subset of test data for speed
            n_shap = min(200, len(self.X_test))
            shap_subset = self.X_test.iloc[:n_shap]
            self.shap_values = self.shap_explainer.shap_values(
                shap_subset.values, nsamples=100
            )
            
            self.shap_expected_value = float(
                self.shap_explainer.expected_value
            ) if not isinstance(self.shap_explainer.expected_value, list) else float(
                self.shap_explainer.expected_value[0]
            )

            self.shap_importance = pd.DataFrame({
                'feature': self.feature_names,
                'mean_abs_shap': np.abs(self.shap_values).mean(axis=0)
            }).sort_values('mean_abs_shap', ascending=False)

            print("Top 5 features (SHAP):")
            for i, (_, r) in enumerate(self.shap_importance.head(5).iterrows(), 1):
                print(f"  {i}. {r['feature'][:25]:25}: {r['mean_abs_shap']:.4f}")
        except Exception as e:
            print(f"SHAP failed: {e}")
            import traceback
            traceback.print_exc()
            self.shap_explainer = None
            self.shap_values = None
            self.shap_importance = None

    # ------------------------------------------------------------------
    # Correlations
    # ------------------------------------------------------------------
    def compute_correlations(self):
        # [FIX 6] Pearson / Spearman are NOT feature importance; they
        # measure linear and monotonic association only.
        print("\n" + "="*60)
        print("PEARSON & SPEARMAN CORRELATIONS WITH TARGET")
        print("="*60)
        try:
            pr, pp, sr, sp = [], [], [], []
            for f in self.feature_names:
                a, b = stats.pearsonr(self.X[f], self.y)
                c, d = stats.spearmanr(self.X[f], self.y)
                pr.append(a); pp.append(b); sr.append(c); sp.append(d)

            self.correlation_df = pd.DataFrame({
                'feature': self.feature_names,
                'pearson_r': pr, 'pearson_p': pp,
                'spearman_r': sr, 'spearman_p': sp,
            })
            self.correlation_df['abs_spearman'] = self.correlation_df['spearman_r'].abs()
            self.correlation_df = self.correlation_df.sort_values('abs_spearman', ascending=False)

            print("Top 5 by |Spearman ρ|:")
            for i, (_, r) in enumerate(self.correlation_df.head(5).iterrows(), 1):
                print(f"  {i}. {r['feature'][:25]:25}: "
                      f"r={r['pearson_r']:+.4f} (p={r['pearson_p']:.2e}), "
                      f"ρ={r['spearman_r']:+.4f} (p={r['spearman_p']:.2e})")
        except Exception as e:
            print(f"Correlation failed: {e}")
            self.correlation_df = None

    # ------------------------------------------------------------------
    # [FIX 1] Relative quartile categorization
    # ------------------------------------------------------------------
    def categorize_risk(self, value):
        """
        Relative categorization within the dataset based on training-set
        quartiles. Categories represent relative position within the
        dataset and are NOT externally validated thresholds.
        """
        q1 = self.quartile_thresholds['q25']
        q2 = self.quartile_thresholds['q50']
        q3 = self.quartile_thresholds['q75']
        if value >= q3:
            return "Q4 (upper quartile)", self.colors['danger']
        elif value >= q2:
            return "Q3 (upper-middle quartile)", self.colors['warning']
        elif value >= q1:
            return "Q2 (lower-middle quartile)", self.colors['accent']
        else:
            return "Q1 (lower quartile)", self.colors['success']

    # ==================================================================
    # GUI
    # ==================================================================
    def create_gui(self):
        self.root = tk.Tk()
        self.root.title("CS (MPa) Predictor - NGBoost")
        self.root.geometry("1500x950")
        self.root.configure(bg=self.colors['background'])
        self.root.update_idletasks()
        x = (self.root.winfo_screenwidth() // 2) - 750
        y = (self.root.winfo_screenheight() // 2) - 475
        self.root.geometry(f'1500x950+{x}+{y}')

        self.setup_styles()

        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill='both', expand=True, padx=10, pady=10)

        self.prediction_tab = self.create_prediction_tab()
        self.analysis_tab = self.create_analysis_tab()
        self.performance_tab = self.create_performance_tab()
        self.history_tab = self.create_history_tab()
        self.model_info_tab = self.create_model_info_tab()

        self.notebook.add(self.prediction_tab, text="Prediction")
        self.notebook.add(self.analysis_tab, text="Analysis")
        self.notebook.add(self.performance_tab, text="Performance")
        self.notebook.add(self.history_tab, text="History")
        self.notebook.add(self.model_info_tab, text="Model Info")

        self.prediction_history = []
        self.load_history()
        print("\nGUI created.")

    def setup_styles(self):
        style = ttk.Style()
        style.theme_use('clam')
        style.configure('Accent.TButton', background=self.colors['accent'],
                       foreground='white', font=('Arial', 11, 'bold'))
        style.configure('Success.TButton', background=self.colors['success'],
                       foreground='white', font=('Arial', 11, 'bold'))
        style.configure('Primary.TButton', background=self.colors['primary'],
                       foreground='white', font=('Arial', 10, 'bold'))
        style.configure('Card.TFrame', background=self.colors['card_bg'],
                       relief='raised', borderwidth=2)

    # ---------- Prediction Tab ----------
    def create_prediction_tab(self):
        tab = ttk.Frame(self.notebook)
        paned = ttk.PanedWindow(tab, orient=tk.HORIZONTAL)
        paned.pack(fill='both', expand=True, padx=10, pady=10)
        left = ttk.Frame(paned); paned.add(left, weight=40)
        right = ttk.Frame(paned); paned.add(right, weight=60)

        tk.Label(left, text="CS (MPa) - FEATURE PARAMETERS",
                font=("Arial", 14, "bold"),
                fg=self.colors['soil_brown']).pack(pady=(0, 5))
        tk.Label(left, text="Enter feature values to predict CS (MPa)",
                font=("Arial", 10)).pack()

        # Quick actions
        qa = ttk.LabelFrame(left, text="Quick Actions (dataset samples)", padding=10)
        qa.pack(fill='x', pady=10)
        row = ttk.Frame(qa); row.pack(fill='x')
        ttk.Button(row, text="Random dataset sample",
                  command=self.fill_random_sample).pack(side='left', padx=2,
                                                        expand=True, fill='x')
        ttk.Button(row, text="Min observed CS",
                  command=self.fill_min_risk).pack(side='left', padx=2,
                                                   expand=True, fill='x')
        ttk.Button(row, text="Max observed CS",
                  command=self.fill_max_risk).pack(side='left', padx=2,
                                                   expand=True, fill='x')
        ttk.Button(row, text="Clear",
                  command=self.clear_inputs).pack(side='left', padx=2,
                                                  expand=True, fill='x')

        # Inputs
        ic = ttk.LabelFrame(left, text="Feature Inputs", padding=10)
        ic.pack(fill='both', expand=True)
        canvas = tk.Canvas(ic, highlightthickness=0)
        sb = ttk.Scrollbar(ic, orient="vertical", command=canvas.yview)
        sf = ttk.Frame(canvas)
        sf.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=sf, anchor="nw")
        canvas.configure(yscrollcommand=sb.set)
        canvas.pack(side="left", fill="both", expand=True)
        sb.pack(side="right", fill="y")

        self.entries = {}
        for f in self.feature_names:
            st = self.feature_stats[f]
            fr = ttk.Frame(sf); fr.pack(fill='x', pady=6)
            tk.Label(fr, text=f, font=("Arial", 9, "bold")).pack(anchor='w')
            r = ttk.Frame(fr); r.pack(fill='x')
            e = ttk.Entry(r, width=15); e.insert(0, f"{st['mean']:.4f}")
            e.pack(side='left')
            tk.Label(r, text=f"[{st['min']:.3f}, {st['max']:.3f}]",
                    font=("Arial", 8)).pack(side='left', padx=5)
            if st['max'] - st['min'] > 0:
                ttk.Scale(r, from_=st['min'], to=st['max'], value=st['mean'],
                         orient='horizontal', length=150,
                         command=lambda v, ff=f: self.update_entry_from_slider(ff, float(v))
                         ).pack(side='right', padx=5)
            self.entries[f] = e

        self.predict_btn = ttk.Button(left, text="PREDICT CS (MPa)",
                                     command=self.threaded_predict,
                                     style='Accent.TButton')
        self.predict_btn.pack(fill='x', pady=10, ipady=8)

        # Right panel
        tk.Label(right, text="PREDICTION RESULT",
                font=("Arial", 14, "bold"),
                fg=self.colors['soil_brown']).pack(pady=(0, 10))
        card = ttk.Frame(right, style='Card.TFrame'); card.pack(fill='x', padx=10, pady=5)
        self.result_var = tk.StringVar(value="---")
        tk.Label(card, textvariable=self.result_var,
                font=("Arial", 40, "bold"),
                fg=self.colors['strength_blue']).pack(pady=15)
        tk.Label(card, text="Predicted CS (MPa)",
                font=("Arial", 12)).pack()
        self.pi_var = tk.StringVar(value="Approximate 95% prediction interval: ---")
        tk.Label(card, textvariable=self.pi_var,
                font=("Arial", 10, "bold"),
                fg=self.colors['secondary']).pack(pady=(0, 15))

        # Categorization
        cat_card = ttk.Frame(right, style='Card.TFrame'); cat_card.pack(fill='x', padx=10, pady=5)
        tk.Label(cat_card, text="Quartile-Based Categorization (relative)",
                font=("Arial", 11, "bold")).pack(pady=(10, 0))
        self.cat_var = tk.StringVar(value="Not predicted")
        self.cat_label = tk.Label(cat_card, textvariable=self.cat_var,
                                 font=("Arial", 18, "bold"))
        self.cat_label.pack(pady=8)
        tk.Label(cat_card,
                text=(f"Q1 < {self.quartile_thresholds['q25']:.3f}  |  "
                      f"Q2 < {self.quartile_thresholds['q50']:.3f}  |  "
                      f"Q3 < {self.quartile_thresholds['q75']:.3f}  |  Q4 ≥ Q3"),
                font=("Arial", 8), fg=self.colors['primary']).pack(pady=(0, 10))
        tk.Label(cat_card,
                text=("Note: categories represent relative position within the dataset "
                      "and are not externally validated thresholds."),
                font=("Arial", 8, "italic"),
                fg=self.colors['primary'], wraplength=500).pack(pady=(0, 10))

        # Detail tabs
        dn = ttk.Notebook(right); dn.pack(fill='both', expand=True, padx=10, pady=10)
        shap_tab = ttk.Frame(dn); dn.add(shap_tab, text="SHAP Contributions")
        self.shap_text = scrolledtext.ScrolledText(shap_tab, font=("Courier", 10),
                                                  wrap=tk.WORD, height=12)
        self.shap_text.pack(fill='both', expand=True, padx=5, pady=5)
        self.shap_text.insert(1.0, "SHAP-based contributions will appear here.")
        self.shap_text.config(state='disabled')

        cmp_tab = ttk.Frame(dn); dn.add(cmp_tab, text="Comparison")
        self.cmp_text = scrolledtext.ScrolledText(cmp_tab, font=("Courier", 10),
                                                 wrap=tk.WORD, height=12)
        self.cmp_text.pack(fill='both', expand=True, padx=5, pady=5)
        self.cmp_text.insert(1.0, "Comparison with typical values will appear here.")
        self.cmp_text.config(state='disabled')

        # Actions
        af = ttk.Frame(right); af.pack(fill='x', padx=10, pady=10)
        for t, c in [("Save Result", self.save_result),
                     ("Copy", self.copy_to_clipboard),
                     ("Export history", self.export_results),
                     ("Analysis tab", lambda: self.notebook.select(1))]:
            ttk.Button(af, text=t, command=c,
                      style='Primary.TButton').pack(side='left', padx=2,
                                                    expand=True, fill='x')
        return tab

    # ---------- Analysis Tab ----------
    def create_analysis_tab(self):
        tab = ttk.Frame(self.notebook)
        ctrl = ttk.LabelFrame(tab, text="Analysis Tools", padding=15)
        ctrl.pack(fill='x', padx=20, pady=10)
        grid = ttk.Frame(ctrl); grid.pack()
        btns = [
            ("Permutation (out-of-sample)", self.plot_feature_importance, self.colors['pso_purple']),
            ("Data distribution", self.plot_data_distribution, self.colors['success']),
            ("Correlation matrix", self.plot_correlation_matrix, self.colors['danger']),
            ("Feature distributions", self.plot_feature_statistics, self.colors['rf_green']),
            ("Observed vs predicted", self.plot_observed_vs_predicted, self.colors['primary']),
            ("Residual analysis", self.plot_residuals, self.colors['soil_brown']),
            ("SHAP summary", self.plot_shap_summary, self.colors['secondary']),
            ("Pearson / Spearman", self.plot_correlation_stats, self.colors['warning']),
        ]
        for i, (t, c, col) in enumerate(btns):
            r, cc = divmod(i, 4)
            tk.Button(grid, text=t, command=c, bg=col, fg='white',
                     font=("Arial", 10, "bold"), padx=10, pady=8, width=24).grid(
                row=r, column=cc, padx=4, pady=4)
        tk.Button(ctrl, text="Save all plots (600 dpi, publication quality)",
                 command=self.save_all_plots,
                 bg=self.colors['soil_brown'], fg='white',
                 font=("Arial", 12, "bold"), padx=20, pady=8).pack(pady=10)
        self.analysis_frame = ttk.Frame(tab)
        self.analysis_frame.pack(fill='both', expand=True, padx=20, pady=10)
        self.plot_feature_importance()
        return tab

    # ---------- Performance Tab ----------
    def create_performance_tab(self):
        tab = ttk.Frame(self.notebook)
        tk.Label(tab, text="PUBLICATION-READY MODEL PERFORMANCE",
                font=("Arial", 14, "bold"),
                fg=self.colors['soil_brown']).pack(pady=10)
        canvas = tk.Canvas(tab, highlightthickness=0)
        sb = ttk.Scrollbar(tab, orient="vertical", command=canvas.yview)
        sf = ttk.Frame(canvas)
        sf.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=sf, anchor="nw")
        canvas.configure(yscrollcommand=sb.set)
        canvas.pack(side="left", fill="both", expand=True)
        sb.pack(side="right", fill="y")

        f1 = ttk.LabelFrame(sf, text="Table 1. Model Performance", padding=10)
        f1.pack(fill='x', padx=20, pady=5)
        self.perf_text = tk.Text(f1, font=("Courier", 10), height=18, wrap='none')
        self.perf_text.pack(fill='x')
        self._populate_performance_table()

        f2 = ttk.LabelFrame(sf, text="Table 2. Pearson & Spearman Correlations", padding=10)
        f2.pack(fill='x', padx=20, pady=5)
        self.corr_text = tk.Text(f2, font=("Courier", 10), height=16, wrap='none')
        self.corr_text.pack(fill='x')
        self._populate_correlation_table()

        f3 = ttk.LabelFrame(sf, text="Table 3. 5-Fold Cross-Validation", padding=10)
        f3.pack(fill='x', padx=20, pady=5)
        self.cv_text = tk.Text(f3, font=("Courier", 10), height=12, wrap='none')
        self.cv_text.pack(fill='x')
        self._populate_cv_table()

        tk.Button(sf, text="Export tables to CSV",
                 command=self.export_performance_tables,
                 bg=self.colors['success'], fg='white',
                 font=("Arial", 12, "bold"), padx=20, pady=8).pack(pady=10)
        return tab

    def _populate_performance_table(self):
        self.perf_text.config(state='normal')
        self.perf_text.delete(1.0, tk.END)
        txt = "="*90 + "\n"
        txt += "  CS (MPa) PREDICTION - NGBOOST MODEL PERFORMANCE\n"
        txt += "="*90 + "\n\n"
        txt += f"  Dataset file:               {os.path.basename(self.data_path)}\n"
        txt += f"  Target variable:            {self.target_name}\n"
        txt += f"  Number of features:         {len(self.feature_names)}\n"
        txt += f"  Total samples:              {len(self.X)}\n"
        txt += f"  Development samples (80%):  {len(self.X_train)}\n"
        txt += f"  Test samples (20%):         {len(self.X_test)} (independent)\n\n"
        txt += "="*90 + "\n"
        txt += f"  {'Metric':<12} {'Train':<14} {'Test':<14} {'Approx. 95% PI (Test)':<26}\n"
        txt += "="*90 + "\n"
        pi_str = f"±{self.pi_half_width:.4f}" if self.pi_half_width else "N/A"
        rows = [
            ('R²',   f"{self.r2_train:.4f}",  f"{self.r2_test:.4f}",  ''),
            ('RMSE', f"{self.rmse_train:.4f}", f"{self.rmse_test:.4f}", pi_str),
            ('MAE',  f"{self.mae_train:.4f}",  f"{self.mae_test:.4f}",  ''),
            ('MAPE', 'N/A', f"{self.mape_test:.2f}%", '(supplementary)'),
        ]
        for n, a, b, c in rows:
            txt += f"  {n:<12} {a:<14} {b:<14} {c:<26}\n"
        if self.cv_results:
            txt += "\n" + "="*90 + "\n"
            txt += "  5-Fold CV (development data): "
            txt += f"R²={self.cv_results['r2_mean']:.4f} ± {self.cv_results['r2_std']:.4f}, "
            txt += f"RMSE={self.cv_results['rmse_mean']:.4f} ± {self.cv_results['rmse_std']:.4f}\n"
            txt += f"  Empirical coverage of approximate 95% PI on test set: "
            txt += f"{self.pi_empirical_coverage:.1f}%\n"
            txt += "  (Empirical coverage is reported for transparency; the interval is "
            txt += "an\n  approximation based on training residual variability and does not\n"
            txt += "  guarantee 95% coverage.)\n"
        txt += "\n" + "="*90 + "\n"
        txt += "  NGBOOST HYPERPARAMETERS\n"
        txt += "="*90 + "\n"
        for k, v in self.hyperparams.items():
            txt += f"  {k:<25}: {v}\n"
        self.perf_text.insert(tk.END, txt)
        self.perf_text.config(state='disabled')

    def _populate_correlation_table(self):
        if self.correlation_df is None:
            return
        self.corr_text.config(state='normal')
        self.corr_text.delete(1.0, tk.END)
        txt = "="*90 + "\n"
        txt += "  PEARSON & SPEARMAN CORRELATIONS WITH CS (MPa)\n"
        txt += "  Note: correlation is NOT feature importance.\n"
        txt += "  *** p<0.001, ** p<0.01, * p<0.05, ns = not significant\n"
        txt += "="*90 + "\n"
        txt += f"  {'Feature':<30} {'Pearson r':>12} {'p':>12} {'Spearman ρ':>14} {'p':>12}\n"
        txt += "="*90 + "\n"
        for _, r in self.correlation_df.iterrows():
            def sig(p):
                if p < 0.001: return "***"
                if p < 0.01:  return "**"
                if p < 0.05:  return "*"
                return "ns"
            txt += (f"  {r['feature'][:30]:<30} {r['pearson_r']:>+12.4f} "
                    f"{r['pearson_p']:>12.2e} {r['spearman_r']:>+14.4f} "
                    f"{r['spearman_p']:>12.2e}  {sig(r['pearson_p'])}\n")
        self.corr_text.insert(tk.END, txt)
        self.corr_text.config(state='disabled')

    def _populate_cv_table(self):
        if not self.cv_results:
            return
        self.cv_text.config(state='normal')
        self.cv_text.delete(1.0, tk.END)
        txt = "="*70 + "\n"
        txt += "  5-FOLD CROSS-VALIDATION (development data)\n"
        txt += "="*70 + "\n"
        txt += f"  {'Fold':<8} {'R²':>12} {'RMSE':>12} {'MAE':>12}\n"
        txt += "="*70 + "\n"
        for i in range(5):
            txt += (f"  {i+1:<8} {self.cv_results['r2_scores'][i]:>12.4f} "
                    f"{self.cv_results['rmse_scores'][i]:>12.4f} "
                    f"{self.cv_results['mae_scores'][i]:>12.4f}\n")
        txt += "-"*70 + "\n"
        txt += (f"  {'Mean':<8} {self.cv_results['r2_mean']:>12.4f} "
                f"{self.cv_results['rmse_mean']:>12.4f} "
                f"{self.cv_results['mae_mean']:>12.4f}\n")
        txt += (f"  {'Std':<8} {self.cv_results['r2_std']:>12.4f} "
                f"{self.cv_results['rmse_std']:>12.4f} "
                f"{self.cv_results['mae_std']:>12.4f}\n")
        txt += "="*70 + "\n"
        self.cv_text.insert(tk.END, txt)
        self.cv_text.config(state='disabled')

    def export_performance_tables(self):
        try:
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            folder = os.path.join(self.save_dir, f"performance_tables_{ts}")
            os.makedirs(folder, exist_ok=True)

            perf_df = pd.DataFrame([
                {'Set': 'Training', 'R2': self.r2_train, 'RMSE': self.rmse_train,
                 'MAE': self.mae_train, 'MAPE_pct': np.nan},
                {'Set': 'Test', 'R2': self.r2_test, 'RMSE': self.rmse_test,
                 'MAE': self.mae_test, 'MAPE_pct': self.mape_test},
                {'Set': 'CV (mean)', 'R2': self.cv_results['r2_mean'],
                 'RMSE': self.cv_results['rmse_mean'], 'MAE': self.cv_results['mae_mean'],
                 'MAPE_pct': np.nan},
                {'Set': 'CV (std)', 'R2': self.cv_results['r2_std'],
                 'RMSE': self.cv_results['rmse_std'], 'MAE': self.cv_results['mae_std'],
                 'MAPE_pct': np.nan},
            ])
            perf_df.to_csv(os.path.join(folder, 'model_performance.csv'), index=False)
            if self.correlation_df is not None:
                self.correlation_df.to_csv(os.path.join(folder, 'correlations.csv'), index=False)
            self.feature_importance.to_csv(os.path.join(folder, 'permutation_importance.csv'), index=False)
            if getattr(self, 'shap_importance', None) is not None:
                self.shap_importance.to_csv(os.path.join(folder, 'shap_importance.csv'), index=False)

            pred_df = pd.DataFrame({
                'y_true': self.y_test.values,
                'y_pred': self.y_test_pred,
                'residual': self.y_test.values - self.y_test_pred,
                'approx_PI_lower_95': self.test_pi_lower,
                'approx_PI_upper_95': self.test_pi_upper,
            })
            if getattr(self, 'y_test_std_native', None) is not None:
                pred_df['ngboost_native_std'] = self.y_test_std_native
            pred_df.to_csv(os.path.join(folder, 'test_predictions.csv'), index=False)
            messagebox.showinfo("Exported", f"Saved to:\n{folder}")
        except Exception as e:
            messagebox.showerror("Export Failed", str(e))

    # ---------- History Tab ----------
    def create_history_tab(self):
        tab = ttk.Frame(self.notebook)
        tk.Label(tab, text="PREDICTION HISTORY",
                font=("Arial", 14, "bold"),
                fg=self.colors['soil_brown']).pack(pady=10)
        ctrl = ttk.LabelFrame(tab, text="History management", padding=10)
        ctrl.pack(fill='x', padx=20, pady=5)
        row = ttk.Frame(ctrl); row.pack()
        for t, c, col in [("Clear", self.clear_history, self.colors['danger']),
                          ("Export CSV", self.export_history, self.colors['success']),
                          ("Load", self.load_history_from_file, self.colors['primary']),
                          ("Refresh", self.update_history_display, self.colors['pso_purple'])]:
            tk.Button(row, text=t, command=c, bg=col, fg='white',
                     font=("Arial", 10, "bold")).pack(side='left', padx=5)

        self.hist_stats = tk.StringVar(value="No predictions yet")
        tk.Label(ctrl, textvariable=self.hist_stats,
                font=("Arial", 11, "bold")).pack(pady=5)

        df = ttk.Frame(tab); df.pack(fill='both', expand=True, padx=20, pady=5)
        cols = ('Timestamp', 'CS (MPa)', 'Categorization', 'Top Feature')
        self.hist_tree = ttk.Treeview(df, columns=cols, show='headings', height=20)
        for c, w in zip(cols, [180, 120, 240, 200]):
            self.hist_tree.heading(c, text=c); self.hist_tree.column(c, width=w)
        vsb = ttk.Scrollbar(df, orient="vertical", command=self.hist_tree.yview)
        self.hist_tree.configure(yscrollcommand=vsb.set)
        self.hist_tree.grid(row=0, column=0, sticky='nsew')
        vsb.grid(row=0, column=1, sticky='ns')
        df.grid_rowconfigure(0, weight=1); df.grid_columnconfigure(0, weight=1)
        return tab

    # ---------- Model Info Tab ----------
    def create_model_info_tab(self):
        tab = ttk.Frame(self.notebook)
        canvas = tk.Canvas(tab); sb = ttk.Scrollbar(tab, orient="vertical", command=canvas.yview)
        sf = ttk.Frame(canvas)
        sf.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=sf, anchor="nw")
        canvas.configure(yscrollcommand=sb.set); canvas.pack(side="left", fill="both", expand=True)
        sb.pack(side="right", fill="y")

        f = ttk.LabelFrame(sf, text="Model Information", padding=15)
        f.pack(fill='x', padx=20, pady=10)

        info = (
            "CS (MPa) PREDICTOR - NGBOOST\n"
            "="*63 + "\n\n"
            f"Target variable:     {self.target_name}\n"
            f"Features:            {len(self.feature_names)}\n"
            f"Development samples: {len(self.X_train)} (80%)\n"
            f"Test samples:        {len(self.X_test)} (20%, independent)\n"
            f"Cross-validation:    5-fold on development data\n"
            "Global importance:   Permutation (out-of-sample, test set)\n"
            "Individual attrib.:  SHAP (KernelExplainer)\n"
            "Uncertainty:         Approximate 95% prediction interval\n"
            "                     based on training residual variability\n"
            "                     + NGBoost native predictive distribution\n\n"
            "NGBOOST HYPERPARAMETERS\n"
            + "-"*63 + "\n"
        )
        for k, v in self.hyperparams.items():
            info += f"  {k:<25}: {v}\n"
        info += (
            "\n" + "="*63 + "\n"
            "QUARTILE-BASED CATEGORIZATION\n"
            "(relative position within the dataset, not externally\n"
            "validated thresholds)\n"
            + "-"*63 + "\n"
            f"  Q1 (< {self.quartile_thresholds['q25']:.4f})\n"
            f"  Q2 ({self.quartile_thresholds['q25']:.4f} - {self.quartile_thresholds['q50']:.4f})\n"
            f"  Q3 ({self.quartile_thresholds['q50']:.4f} - {self.quartile_thresholds['q75']:.4f})\n"
            f"  Q4 (>= {self.quartile_thresholds['q75']:.4f})\n\n"
            "Interpretation note: These quartiles are computed on the\n"
            "training-set distribution only. They are a relative ordinal\n"
            "categorization and are NOT externally validated thresholds.\n"
        )
        tk.Label(f, text=info, font=("Courier", 10), justify='left',
                bg='white', padx=15, pady=15).pack(fill='x')
        return tab

    # ==================================================================
    # Prediction
    # ==================================================================
    def threaded_predict(self):
        self.predict_btn.config(state='disabled', text="Predicting...")
        t = threading.Thread(target=self.predict_risk); t.daemon = True; t.start()

    def predict_risk(self):
        try:
            # [FIX 8] Build the input dict and PASS IT to history
            inputs = {}
            for f, e in self.entries.items():
                try:
                    inputs[f] = float(e.get().strip())
                except:
                    inputs[f] = float(self.feature_stats[f]['mean'])

            X_in = pd.DataFrame([inputs], columns=self.feature_names)
            prediction = float(self.model.predict(X_in.values)[0])

            # Approximate 95% prediction interval
            if self.pi_half_width is not None:
                pi_lo = prediction - self.pi_half_width
                pi_hi = prediction + self.pi_half_width
            else:
                pi_lo = pi_hi = None

            # NGBoost native uncertainty
            native_std = None
            try:
                dist = self.model.pred_dist(X_in.values)
                native_std = float(dist.params['scale'][0])
            except Exception as e:
                print(f"NGBoost native uncertainty failed: {e}")

            # SHAP per-instance
            shap_c = {}
            if getattr(self, 'shap_explainer', None) is not None:
                try:
                    sv = self.shap_explainer.shap_values(X_in.values, nsamples=100)
                    if isinstance(sv, list): sv = sv[0]
                    sv = np.array(sv).flatten()
                    for i, f in enumerate(self.feature_names):
                        shap_c[f] = float(sv[i])
                except Exception as e:
                    print(f"SHAP per-instance failed: {e}")

            # comparison list
            comp = [{'feature': f, 'value': inputs[f],
                     'mean': self.feature_stats[f]['mean']}
                    for f in self.feature_importance.head(5)['feature']
                    if f in inputs]

            self.root.after(0, lambda: self.display_result(
                prediction, pi_lo, pi_hi, shap_c, comp, inputs, native_std))
        except Exception as e:
            import traceback
            traceback.print_exc()
            self.root.after(0, lambda: messagebox.showerror("Prediction Error", str(e)))
        self.root.after(0, self.enable_predict_button)

    def enable_predict_button(self):
        self.predict_btn.config(state='normal', text="PREDICT CS (MPa)")

    def display_result(self, prediction, pi_lo, pi_hi, shap_c, comp, inputs, native_std=None):
        self.result_var.set(f"{prediction:.4f}")
        if pi_lo is not None:
            pi_str = f"Approximate 95% prediction interval: [{pi_lo:.4f}, {pi_hi:.4f}]"
            if native_std is not None:
                pi_str += f"  |  NGBoost native std: {native_std:.4f}"
            self.pi_var.set(pi_str)
        else:
            self.pi_var.set("Approximate 95% prediction interval: N/A")

        cat, col = self.categorize_risk(prediction)
        self.cat_var.set(cat); self.cat_label.config(fg=col)

        self.update_shap_text(shap_c, prediction)
        self.update_comparison_text(comp, prediction)

        # [FIX 8] Pass the ACTUAL input dictionary
        self.save_to_history(prediction, cat, inputs=inputs)

    def update_shap_text(self, shap_c, prediction):
        self.shap_text.config(state='normal'); self.shap_text.delete(1.0, tk.END)
        if not shap_c:
            self.shap_text.insert(1.0, "SHAP not available. Install with: pip install shap")
            self.shap_text.config(state='disabled'); return
        base = self.shap_expected_value
        txt = "SHAP INDIVIDUAL CONTRIBUTIONS\n"
        txt += "="*60 + "\n"
        txt += f"Base value (E[f(X)]): {base:.4f}\n"
        txt += f"Prediction:           {prediction:.4f}\n"
        txt += f"Sum(base + SHAP):     {base + sum(shap_c.values()):.4f}\n\n"
        for f, v in sorted(shap_c.items(), key=lambda x: abs(x[1]), reverse=True):
            d = "increases" if v > 0 else "decreases"
            b = int(min(abs(v)*30, 20))
            txt += f"  {f[:28]:28} {d:10} CS  |{'█'*b + '░'*(20-b)}| {v:+.4f}\n"
        self.shap_text.insert(1.0, txt)
        self.shap_text.config(state='disabled')

    def update_comparison_text(self, comp, prediction):
        self.cmp_text.config(state='normal'); self.cmp_text.delete(1.0, tk.END)
        txt = "COMPARISON WITH TYPICAL VALUES (training-set means)\n"
        txt += "="*60 + "\n\n"
        for c in comp:
            diff = c['value'] - c['mean']
            pct = (diff/c['mean']*100) if c['mean'] else 0
            txt += f"{c['feature'][:30]}\n"
            txt += f"  Current: {c['value']:.4f} | Typical: {c['mean']:.4f}"
            if abs(diff) > 1e-9:
                txt += f"  ({'+' if diff>0 else ''}{pct:.1f}%)\n"
            else:
                txt += "\n"
            txt += "\n"
        txt += "\nCATEGORIZATION (relative, quartile-based)\n"
        txt += "="*60 + "\n"
        cat, _ = self.categorize_risk(prediction)
        txt += f"Assigned category: {cat}\n\n"
        txt += "Quartile thresholds (from training set):\n"
        txt += f"  Q1 < {self.quartile_thresholds['q25']:.4f}\n"
        txt += f"  Q2 < {self.quartile_thresholds['q50']:.4f}\n"
        txt += f"  Q3 < {self.quartile_thresholds['q75']:.4f}\n"
        txt += f"  Q4 ≥ {self.quartile_thresholds['q75']:.4f}\n\n"
        txt += ("Note: These categories describe relative position within "
                "the dataset.\nThey are not externally validated thresholds.\n")
        self.cmp_text.insert(1.0, txt)
        self.cmp_text.config(state='disabled')

    # ------------------------------------------------------------------
    # History
    # ------------------------------------------------------------------
    def save_to_history(self, prediction, categorization, inputs):
        top_f = self.feature_importance.iloc[0]['feature']
        entry = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'CS_MPa': float(prediction),
            'categorization': categorization,
            'model': 'NGBoost_FixedParams',
            'importance_type': 'Permutation(out-of-sample)+SHAP',
            'top_feature': top_f,
        }
        # [FIX 8] Store every input value
        for f, v in inputs.items():
            entry[f] = float(v)
        self.prediction_history.append(entry)
        self.update_history_display()
        self.save_history()

    def update_history_display(self):
        for i in self.hist_tree.get_children():
            self.hist_tree.delete(i)
        for e in reversed(self.prediction_history[-100:]):
            self.hist_tree.insert('', 'end', values=(
                e['timestamp'], f"{e.get('CS_MPa', e.get('risk_index', 0)):.4f}",
                e.get('categorization', ''), e.get('top_feature', 'N/A')))
        if self.prediction_history:
            v = [h.get('CS_MPa', h.get('risk_index', 0)) for h in self.prediction_history]
            self.hist_stats.set(
                f"Total: {len(v)} | Mean: {np.mean(v):.4f} | "
                f"Range: {min(v):.4f} - {max(v):.4f}")

    def clear_history(self):
        if not self.prediction_history:
            messagebox.showinfo("History", "Empty."); return
        if messagebox.askyesno("Clear", "Clear all history?"):
            self.prediction_history = []
            self.update_history_display(); self.save_history()

    def export_history(self):
        if not self.prediction_history:
            messagebox.showwarning("History", "Nothing to export."); return
        fn = filedialog.asksaveasfilename(defaultextension=".csv",
            initialfile=f"predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        if fn:
            pd.DataFrame(self.prediction_history).to_csv(fn, index=False)
            messagebox.showinfo("Exported", fn)

    def load_history_from_file(self):
        fn = filedialog.askopenfilename(filetypes=[("JSON", "*.json"), ("All", "*.*")])
        if fn:
            with open(fn) as fh:
                self.prediction_history = json.load(fh)
            self.update_history_display()

    def save_history(self):
        try:
            with open(self.history_file, 'w') as fh:
                json.dump(self.prediction_history, fh, indent=2)
        except Exception as e:
            print(f"History save failed: {e}")

    def load_history(self):
        try:
            if os.path.exists(self.history_file):
                with open(self.history_file) as fh:
                    self.prediction_history = json.load(fh)
                self.update_history_display()
        except:
            pass

    # ------------------------------------------------------------------
    # Save result / clipboard
    # ------------------------------------------------------------------
    def save_result(self):
        txt = "CS (MPa) PREDICTION - NGBOOST\n"
        txt += "="*60 + "\n"
        txt += f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        txt += f"Model: NGBoost (fixed hyperparameters)\n"
        txt += f"Global importance: Permutation (out-of-sample)\n"
        txt += f"Individual attribution: SHAP\n"
        txt += f"Data split: 80% development / 20% independent test\n"
        txt += f"Cross-validation: 5-fold on development data\n\n"
        txt += f"Predicted CS (MPa):  {self.result_var.get()}\n"
        txt += f"{self.pi_var.get()}\n"
        txt += f"Quartile-based categorization: {self.cat_var.get()}\n\n"
        txt += "Categorization thresholds (relative to training distribution):\n"
        txt += f"  Q1 < {self.quartile_thresholds['q25']:.4f}\n"
        txt += f"  Q2 < {self.quartile_thresholds['q50']:.4f}\n"
        txt += f"  Q3 < {self.quartile_thresholds['q75']:.4f}\n"
        txt += f"  Q4 ≥ {self.quartile_thresholds['q75']:.4f}\n\n"
        txt += ("Note: Categorization is relative to the dataset distribution "
                "and is\nnot an externally validated threshold.\n\n")
        txt += "Input parameters:\n"
        for f, e in self.entries.items():
            txt += f"  {f:30}: {e.get()}\n"
        fn = filedialog.asksaveasfilename(
            defaultextension=".txt",
            initialfile=f"CS_prediction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
        if fn:
            with open(fn, 'w') as fh:
                fh.write(txt)
            messagebox.showinfo("Saved", fn)

    def copy_to_clipboard(self):
        self.root.clipboard_clear()
        self.root.clipboard_append(
            f"CS (MPa): {self.result_var.get()} | {self.pi_var.get()} | {self.cat_var.get()}")
        messagebox.showinfo("Copied", "Result copied.")

    def export_results(self):
        self.export_history()

    # ------------------------------------------------------------------
    # Utility
    # ------------------------------------------------------------------
    def fill_random_sample(self):
        idx = np.random.randint(0, len(self.df))
        s = self.df.iloc[idx]
        for f in self.entries:
            if f in s:
                self.entries[f].delete(0, tk.END)
                self.entries[f].insert(0, f"{s[f]:.4f}")
        messagebox.showinfo("Random dataset sample",
            f"Dataset row #{idx}\nObserved CS (MPa): {s[self.target_name]:.4f}")

    def fill_min_risk(self):
        idx = self.y.idxmin(); s = self.df.iloc[idx]
        for f in self.entries:
            if f in s:
                self.entries[f].delete(0, tk.END)
                self.entries[f].insert(0, f"{s[f]:.4f}")
        messagebox.showinfo("Minimum observed CS (MPa)",
            f"Observed minimum: {self.y.min():.4f}")

    def fill_max_risk(self):
        idx = self.y.idxmax(); s = self.df.iloc[idx]
        for f in self.entries:
            if f in s:
                self.entries[f].delete(0, tk.END)
                self.entries[f].insert(0, f"{s[f]:.4f}")
        messagebox.showinfo("Maximum observed CS (MPa)",
            f"Observed maximum: {self.y.max():.4f}")

    def update_entry_from_slider(self, f, v):
        if f in self.entries:
            self.entries[f].delete(0, tk.END)
            self.entries[f].insert(0, f"{v:.4f}")

    def clear_inputs(self):
        for f, e in self.entries.items():
            e.delete(0, tk.END); e.insert(0, f"{self.feature_stats[f]['mean']:.4f}")
        self.result_var.set("---")
        self.pi_var.set("Approximate 95% prediction interval: ---")
        self.cat_var.set("Not predicted")

    # ==================================================================
    # PLOTS - publication quality (600 dpi, Times New Roman)
    # ==================================================================
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'DejaVu Serif'],
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 13,
        'axes.linewidth': 1.0,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'savefig.dpi': 600,
        'figure.dpi': 100,
    })

    def clear_analysis_frame(self):
        for w in self.analysis_frame.winfo_children():
            w.destroy()

    def _embed(self, fig):
        canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill='both', expand=True)

    def save_plot(self, func, name):
        fig = func(save_mode=True)
        if fig:
            path = os.path.join(self.save_dir,
                f"{name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png")
            fig.savefig(path, dpi=600, bbox_inches='tight')
            plt.close(fig)
            print(f"Saved: {path}")
            return True
        return False

    def save_all_plots(self):
        try:
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            folder = os.path.join(self.save_dir, f"plots_{ts}")
            os.makedirs(folder, exist_ok=True)
            orig = self.save_dir; self.save_dir = folder
            funcs = [
                (self.plot_feature_importance, "permutation_importance"),
                (self.plot_data_distribution, "data_distribution"),
                (self.plot_correlation_matrix, "correlation_matrix"),
                (self.plot_feature_statistics, "feature_distributions"),
                (self.plot_observed_vs_predicted, "observed_vs_predicted"),
                (self.plot_residuals, "residual_analysis"),
                (self.plot_shap_summary, "shap_summary"),
                (self.plot_correlation_stats, "correlation_stats"),
            ]
            saved = []
            for f, n in funcs:
                try:
                    if self.save_plot(f, n): saved.append(n)
                except Exception as e:
                    print(f"{n} failed: {e}")
            self.save_dir = orig
            messagebox.showinfo("Plots Saved",
                f"Saved {len(saved)} plots to:\n{folder}")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    def plot_feature_importance(self, save_mode=False):
        if not save_mode: self.clear_analysis_frame()
        fig = Figure(figsize=(10, 7), facecolor='white')
        ax = fig.add_subplot(111)
        d = self.feature_importance.head(15)
        y = np.arange(len(d))
        ax.barh(y, d['importance'], color='#CC7722',
               edgecolor='black', linewidth=0.6,
               xerr=d['std'], capsize=3)
        ax.set_yticks(y); ax.set_yticklabels(d['feature'])
        ax.set_xlabel('Permutation importance (mean decrease in R²)')
        ax.set_title('Out-of-sample permutation feature importance (test set)')
        ax.grid(True, alpha=0.3, axis='x', linestyle='--')
        ax.invert_yaxis()
        for i, (_, r) in enumerate(d.iterrows()):
            ax.text(r['importance'] + r['std'] + 1e-4, i,
                   f"{r['importance']:.4f} ± {r['std']:.4f}",
                   va='center', fontsize=8)
        fig.tight_layout()
        if not save_mode: self._embed(fig)
        return fig

    def plot_data_distribution(self, save_mode=False):
        if not save_mode: self.clear_analysis_frame()
        fig = Figure(figsize=(10, 6), facecolor='white')
        ax = fig.add_subplot(111)
        ax.hist(self.y, bins=30, color='#D4A373', edgecolor='black',
               linewidth=0.5, density=True)
        for q, lbl, col in [(self.quartile_thresholds['q25'], 'Q1', '#2D6A4F'),
                            (self.quartile_thresholds['q50'], 'Q2 (median)', '#B9770E'),
                            (self.quartile_thresholds['q75'], 'Q3', '#B03A2E')]:
            ax.axvline(q, color=col, linestyle='--', linewidth=1.5,
                      label=f'{lbl} = {q:.3f}')
        ax.set_xlabel('CS (MPa)')
        ax.set_ylabel('Probability density')
        ax.set_title('Distribution of CS (MPa) with quartile thresholds')
        ax.legend(); ax.grid(True, alpha=0.3, linestyle='--')
        fig.tight_layout()
        if not save_mode: self._embed(fig)
        return fig

    def plot_correlation_matrix(self, save_mode=False):
        if not save_mode: self.clear_analysis_frame()
        fig = Figure(figsize=(10, 9), facecolor='white')
        ax = fig.add_subplot(111)
        top = self.feature_importance.head(10)['feature'].tolist()
        feats = top + [self.target_name]
        cm_ = self.df[feats].corr()
        im = ax.imshow(cm_, cmap='RdYlBu_r', vmin=-1, vmax=1, aspect='auto')
        ax.set_xticks(np.arange(len(feats)))
        ax.set_yticks(np.arange(len(feats)))
        ax.set_xticklabels([f[:14] for f in feats], rotation=45, ha='right')
        ax.set_yticklabels([f[:14] for f in feats])
        ax.set_title('Correlation matrix (top features and target)')
        for i in range(len(feats)):
            for j in range(len(feats)):
                ax.text(j, i, f'{cm_.iloc[i,j]:.2f}',
                       ha='center', va='center', fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                    label='Pearson correlation')
        fig.tight_layout()
        if not save_mode: self._embed(fig)
        return fig

    def plot_feature_statistics(self, save_mode=False):
        if not save_mode: self.clear_analysis_frame()
        fig = Figure(figsize=(11, 7), facecolor='white')
        top = self.feature_importance.head(6)['feature'].tolist()
        for i, f in enumerate(top, 1):
            ax = fig.add_subplot(2, 3, i)
            bp = ax.boxplot(self.X[f].dropna(), patch_artist=True, widths=0.55)
            for b in bp['boxes']:
                b.set_facecolor('#D4A373'); b.set_edgecolor('black')
            ax.set_title(f[:18]); ax.grid(True, alpha=0.3, axis='y')
        fig.suptitle('Distributions of top 6 features')
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        if not save_mode: self._embed(fig)
        return fig

    def plot_observed_vs_predicted(self, save_mode=False):
        if not save_mode: self.clear_analysis_frame()
        fig = Figure(figsize=(11, 8), facecolor='white')
        ax1 = fig.add_subplot(2, 2, 1)
        ax1.scatter(self.y_train, self.y_train_pred, s=12, alpha=0.6,
                   edgecolors='black', linewidths=0.3, color='#D4A373')
        lims = [min(self.y_train.min(), self.y_train_pred.min()),
                max(self.y_train.max(), self.y_train_pred.max())]
        ax1.plot(lims, lims, 'k--', linewidth=1)
        ax1.set_xlabel('Observed'); ax1.set_ylabel('Predicted')
        ax1.set_title(f'Development set (R² = {self.r2_train:.3f})')
        ax1.grid(True, alpha=0.3, linestyle='--')

        ax2 = fig.add_subplot(2, 2, 2)
        ax2.errorbar(self.y_test, self.y_test_pred,
                    yerr=self.pi_half_width if self.pi_half_width else 0,
                    fmt='o', markersize=3, alpha=0.6, ecolor='gray',
                    capsize=2, markeredgecolor='black',
                    markerfacecolor='#2D6A4F')
        lims2 = [min(self.y_test.min(), self.y_test_pred.min()),
                 max(self.y_test.max(), self.y_test_pred.max())]
        ax2.plot(lims2, lims2, 'k--', linewidth=1)
        ax2.set_xlabel('Observed'); ax2.set_ylabel('Predicted')
        ax2.set_title(f'Test set (R² = {self.r2_test:.3f}) with approx. 95% PI')
        ax2.grid(True, alpha=0.3, linestyle='--')

        ax3 = fig.add_subplot(2, 1, 2)
        ax3.scatter(self.y_train, self.y_train_pred, s=10, alpha=0.5,
                   label=f'Development (R² = {self.r2_train:.3f})',
                   color='#D4A373', edgecolors='black', linewidths=0.3)
        ax3.scatter(self.y_test, self.y_test_pred, s=12, alpha=0.7,
                   label=f'Test (R² = {self.r2_test:.3f})',
                   color='#2D6A4F', edgecolors='black', linewidths=0.3)
        mn = min(self.y.min(), min(self.y_train_pred.min(), self.y_test_pred.min()))
        mx = max(self.y.max(), max(self.y_train_pred.max(), self.y_test_pred.max()))
        ax3.plot([mn, mx], [mn, mx], 'k--', linewidth=1, label='1:1')
        ax3.set_xlabel('Observed CS (MPa)')
        ax3.set_ylabel('Predicted CS (MPa)')
        ax3.set_title('Observed vs predicted')
        ax3.legend(); ax3.grid(True, alpha=0.3, linestyle='--')
        fig.tight_layout()
        if not save_mode: self._embed(fig)
        return fig

    def plot_residuals(self, save_mode=False):
        if not save_mode: self.clear_analysis_frame()
        fig = Figure(figsize=(12, 9), facecolor='white')
        rtr = self.y_train.values - self.y_train_pred
        rte = self.y_test.values - self.y_test_pred

        ax1 = fig.add_subplot(2, 2, 1)
        ax1.scatter(self.y_train_pred, rtr, s=10, alpha=0.5,
                   color='#D4A373', edgecolors='black', linewidths=0.3, label='Development')
        ax1.scatter(self.y_test_pred, rte, s=12, alpha=0.7,
                   color='#2D6A4F', edgecolors='black', linewidths=0.3, label='Test')
        ax1.axhline(0, color='red', linestyle='--', linewidth=1)
        ax1.set_xlabel('Predicted'); ax1.set_ylabel('Residual')
        ax1.set_title('Residuals vs predicted'); ax1.legend()
        ax1.grid(True, alpha=0.3, linestyle='--')

        ax2 = fig.add_subplot(2, 2, 2)
        ax2.hist(rtr, bins=25, alpha=0.6, label='Development',
                color='#D4A373', edgecolor='black', linewidth=0.4)
        ax2.hist(rte, bins=25, alpha=0.6, label='Test',
                color='#2D6A4F', edgecolor='black', linewidth=0.4)
        ax2.axvline(0, color='red', linestyle='--', linewidth=1)
        ax2.set_xlabel('Residual'); ax2.set_ylabel('Frequency')
        ax2.set_title('Residual distribution'); ax2.legend()
        ax2.grid(True, alpha=0.3, linestyle='--')

        ax3 = fig.add_subplot(2, 2, 3)
        stats.probplot(rte, dist="norm", plot=ax3)
        ax3.set_title('Q-Q plot of test residuals'); ax3.grid(True, alpha=0.3)

        ax4 = fig.add_subplot(2, 2, 4)
        ax4.scatter(self.y_train, rtr, s=10, alpha=0.5,
                   color='#D4A373', edgecolors='black', linewidths=0.3, label='Development')
        ax4.scatter(self.y_test, rte, s=12, alpha=0.7,
                   color='#2D6A4F', edgecolors='black', linewidths=0.3, label='Test')
        ax4.axhline(0, color='red', linestyle='--', linewidth=1)
        ax4.set_xlabel('Observed'); ax4.set_ylabel('Residual')
        ax4.set_title('Residuals vs observed'); ax4.legend()
        ax4.grid(True, alpha=0.3, linestyle='--')

        fig.suptitle('Residual analysis')
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        if not save_mode: self._embed(fig)
        return fig

    def plot_shap_summary(self, save_mode=False):
        if not save_mode: self.clear_analysis_frame()
        if getattr(self, 'shap_values', None) is None:
            fig = Figure(figsize=(8, 4), facecolor='white')
            ax = fig.add_subplot(111); ax.axis('off')
            ax.text(0.5, 0.5, "SHAP values not available.",
                   ha='center', va='center', fontsize=12)
            if not save_mode: self._embed(fig)
            return fig
        fig = Figure(figsize=(10, 8), facecolor='white')
        ax1 = fig.add_subplot(2, 1, 1)
        top = self.shap_importance.head(15)
        y = np.arange(len(top))
        ax1.barh(y, top['mean_abs_shap'], color='#800080', edgecolor='black', linewidth=0.5)
        ax1.set_yticks(y); ax1.set_yticklabels(top['feature'])
        ax1.set_xlabel('Mean |SHAP value|')
        ax1.set_title('SHAP global importance (mean |SHAP|)')
        ax1.invert_yaxis(); ax1.grid(True, alpha=0.3, axis='x', linestyle='--')

        ax2 = fig.add_subplot(2, 1, 2)
        top_idx = [self.feature_names.index(f) for f in top['feature'][:10]]
        for i, fi in enumerate(top_idx):
            vals = self.shap_values[:, fi]
            fv = self.X_test.iloc[:len(vals), fi].values
            fvn = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)
            ax2.scatter(vals, np.full(len(vals), i) + np.random.normal(0, 0.08, len(vals)),
                       c=fvn, cmap='coolwarm', s=12, alpha=0.6, edgecolors='none')
        ax2.set_yticks(range(len(top_idx)))
        ax2.set_yticklabels([self.feature_names[i] for i in top_idx])
        ax2.axvline(0, color='black', linewidth=0.8, linestyle='--')
        ax2.set_xlabel('SHAP value (impact on model output)')
        ax2.set_title('SHAP beeswarm (top 10 features)')
        ax2.grid(True, alpha=0.3, axis='x', linestyle='--')
        fig.tight_layout()
        if not save_mode: self._embed(fig)
        return fig

    def plot_correlation_stats(self, save_mode=False):
        if not save_mode: self.clear_analysis_frame()
        if self.correlation_df is None:
            return None
        fig = Figure(figsize=(11, 7), facecolor='white')
        top = self.correlation_df.head(15)
        y = np.arange(len(top))

        ax1 = fig.add_subplot(1, 2, 1)
        colors = ['#2D6A4F' if r > 0 else '#B03A2E' for r in top['pearson_r']]
        ax1.barh(y, top['pearson_r'], color=colors, edgecolor='black', linewidth=0.5)
        ax1.set_yticks(y); ax1.set_yticklabels(top['feature'])
        ax1.axvline(0, color='black', linewidth=0.8)
        ax1.set_xlabel('Pearson r'); ax1.set_title('Pearson correlation')
        ax1.grid(True, alpha=0.3, axis='x', linestyle='--'); ax1.invert_yaxis()

        ax2 = fig.add_subplot(1, 2, 2)
        colors = ['#2D6A4F' if r > 0 else '#B03A2E' for r in top['spearman_r']]
        ax2.barh(y, top['spearman_r'], color=colors, edgecolor='black', linewidth=0.5)
        ax2.set_yticks(y); ax2.set_yticklabels(top['feature'])
        ax2.axvline(0, color='black', linewidth=0.8)
        ax2.set_xlabel('Spearman ρ'); ax2.set_title('Spearman correlation')
        ax2.grid(True, alpha=0.3, axis='x', linestyle='--'); ax2.invert_yaxis()

        fig.suptitle('Pearson and Spearman correlations with CS (MPa)')
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        if not save_mode: self._embed(fig)
        return fig

    # ==================================================================
    def run(self):
        print("\n" + "="*70)
        print("CS (MPa) PREDICTOR - NGBOOST")
        print("="*70)
        print(f"Target: {self.target_name}")
        print(f"Features: {len(self.feature_names)}")
        print(f"Split: 80% development / 20% independent test")
        print(f"CV: 5-fold on development data")
        print(f"SHAP: {'available' if SHAP_AVAILABLE else 'not installed'}")
        print(f"Output directory: {self.save_dir}")
        print("="*70)
        self.root.mainloop()


if __name__ == "__main__":
    try:
        app = TargetPredictor()
        app.run()
    except KeyboardInterrupt:
        print("\nTerminated by user")
    except Exception as e:
        print(f"Error: {e}")
        import traceback; traceback.print_exc()

Loading data from: D:\2026 Work\My Papers\Gulzar\Data\Data.csv
Dataset shape: (299, 17)
Target variable: 'CS_MPa'
Features: 16
Target stats: mean=60.6765, std=41.3250, min=0.0000, max=153.4000

TRAIN-TEST SPLIT (80-20)
Training: 239 samples (80%)
Test:     60 samples (20%, independent)

Quartile-based Categorization (training data):
  Q1 (25th percentile): 26.1000
  Q2 (median, 50th):    49.8070
  Q3 (75th percentile): 93.5000
  NOTE: These are RELATIVE categories within the dataset,
        not externally validated thresholds.

TRAINING NGBOOST MODEL
  verbose_eval             : False
  verbose                  : False
  tol                      : 1e-05
  natural_gradient         : True
  n_estimators             : 700
  minibatch_frac           : 0.7
  learning_rate            : 0.07
  col_sample               : 1.0
  random_state             : 42
Model trained.

Computing permutation feature importance (out-of-sample, test set)...

Top 5 features (permutation, out-of-sample):
  1. F

  0%|          | 0/60 [00:00<?, ?it/s]

Top 5 features (SHAP):
  1. W_B                      : 18.9271
  2. W_kg_m3                  : 9.2853
  3. SF_kg_m3                 : 8.3983
  4. SP_kg_m3                 : 4.8376
  5. LD_x_y_z                 : 2.3252

PEARSON & SPEARMAN CORRELATIONS WITH TARGET
Top 5 by |Spearman ρ|:
  1. SF_kg_m3                 : r=+0.9391 (p=8.48e-140), ρ=+0.9160 (p=8.00e-120)
  2. W_B                      : r=-0.8436 (p=3.36e-82), ρ=-0.8545 (p=1.74e-86)
  3. W_kg_m3                  : r=-0.8238 (p=3.36e-75), ρ=-0.6947 (p=2.13e-44)
  4. FA_kg_m3                 : r=-0.5874 (p=4.05e-29), ρ=-0.5744 (p=1.20e-27)
  5. SP_kg_m3                 : r=+0.4776 (p=1.92e-18), ρ=+0.5518 (p=3.21e-25)

GUI created.

CS (MPa) PREDICTOR - NGBOOST
Target: CS_MPa
Features: 16
Split: 80% development / 20% independent test
CV: 5-fold on development data
SHAP: available
Output directory: D:\2026 Work\My Papers\Gulzar\GUI


  0%|          | 0/1 [00:00<?, ?it/s]